# HRD Dataset: MinuteLevel Dataset from Raw Cleaned Data

**Purpose**: Build the complete HRD_RAW_MinuteLevel.csv dataset from raw files

**Process**:
1. Load all Fitbit sensor data (HR, Steps, Floors, Activity levels) at minute resolution
2. Merge sleep data from Sleep_Intraday records
3. Load and merge AWARE data (Calls, Screen events)
4. Calculate depression labels from CES-D surveys (Pre/Post/Category)
5. Generate comprehensive analysis and summary reports

**Output**: Single merged CSV with all data at minute level + comprehensive summary reports

## Step 1: Initialize Paths and Load Required Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Setup paths
HRD_DIR = Path(r"c:\Users\umroot\Desktop\Human-Rhythms-Dataset\HRD")
CLEANED_RAW = HRD_DIR / "Cleaned_Raw_Data"
FITBIT_DIR = CLEANED_RAW / "Fitbit"
AWARE_DIR = CLEANED_RAW / "Aware"
SURVEY_DIR = CLEANED_RAW / "Surveys"

OUTPUT_FILE = HRD_DIR / "HRD_RAW_MinuteLevel.csv"

print("="*100)
print("HRD DATASET REBUILD - COMPLETE PIPELINE")
print("="*100)
print(f"\nData source: {CLEANED_RAW}")
print(f"Output: {OUTPUT_FILE}")
print(f"\nDirectory validation:")
print(f"  ✓ Fitbit: {FITBIT_DIR.exists()}")
print(f"  ✓ AWARE: {AWARE_DIR.exists()}")
print(f"  ✓ Surveys: {SURVEY_DIR.exists()}")

HRD DATASET REBUILD - COMPLETE PIPELINE

Data source: c:\Users\umroot\Desktop\Human-Rhythms-Dataset\HRD\Cleaned_Raw_Data
Output: c:\Users\umroot\Desktop\Human-Rhythms-Dataset\HRD\HRD_RAW_MinuteLevel.csv

Directory validation:
  ✓ Fitbit: True
  ✓ AWARE: True
  ✓ Surveys: True


## Step 2: Define Utility Functions for Data Processing

In [2]:
def parse_datetime_to_minute(dt_series):
    """Parse datetime to minute resolution, handling various formats."""
    if dt_series is None or len(dt_series) == 0:
        return pd.Series(dtype='datetime64[ns]')
    
    # Try parsing as datetime
    if pd.api.types.is_datetime64_any_dtype(dt_series):
        dt = pd.to_datetime(dt_series, errors='coerce')
    else:
        s = dt_series.astype(str).str.strip()
        s = s.replace({'': pd.NA, 'nan': pd.NA, 'NaT': pd.NA})
        
        # Try parsing as normal datetime strings
        dt = pd.to_datetime(s, errors='coerce', cache=True)
        
        # Also try numeric epochs if present
        num = pd.to_numeric(s, errors='coerce')
        if num.notna().any():
            med = float(num.dropna().median())
            unit = 'ms' if med > 1e11 else 's'
            dt_num = pd.to_datetime(num, unit=unit, errors='coerce')
            dt = dt.fillna(dt_num)
    
    # Remove timezone if present
    if hasattr(dt.dt, 'tz') and dt.dt.tz is not None:
        dt = dt.dt.tz_localize(None)
    
    # Floor to minute
    return dt.dt.floor('min')

def clean_pid(value):
    """Clean participant ID (standardize to lowercase)."""
    return str(value).strip().lower()

def load_fitbit_csv(path, value_col_name, pid_col='device_id', time_col='local_date_time', value_col='value'):
    """Load Fitbit CSV and return cleaned dataframe with pid, dateTime, value."""
    if not path.exists():
        return pd.DataFrame(columns=['pid', 'dateTime', value_col_name])
    
    try:
        df = pd.read_csv(path)
        
        # Handle column name variations
        if pid_col not in df.columns:
            pid_col = 'pid' if 'pid' in df.columns else df.columns[0]
        if time_col not in df.columns:
            time_col = 'dateTime' if 'dateTime' in df.columns else [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()][0]
        if value_col not in df.columns:
            value_col = [c for c in df.columns if 'value' in c.lower() or c not in [pid_col, time_col]][0]
        
        df['pid'] = df[pid_col].apply(clean_pid)
        df['dateTime'] = parse_datetime_to_minute(df[time_col])
        df[value_col_name] = pd.to_numeric(df[value_col], errors='coerce')
        
        # Keep only rows with valid datetime
        df = df.dropna(subset=['dateTime'])
        df = df[['pid', 'dateTime', value_col_name]]
        
        return df
    except Exception as e:
        print(f"  Error loading {path.name}: {e}")
        return pd.DataFrame(columns=['pid', 'dateTime', value_col_name])

print("✓ Helper functions defined")

✓ Helper functions defined


## Step 3: Load and Merge Fitbit Sensor Data at Minute Resolution

In [3]:
print("\n" + "="*100)
print("STEP 1: LOADING FITBIT SENSOR DATA AT MINUTE RESOLUTION")
print("="*100)

# Define Fitbit source directories
fitbit_sources = {
    'Steps': FITBIT_DIR / 'Steps',
    'Floors': FITBIT_DIR / 'Floors',
    'HR': FITBIT_DIR / 'Heart_Rate_Intraday',
    'Fairly_Active': FITBIT_DIR / 'Minutes_Fairly_Active',
    'Lightly_Active': FITBIT_DIR / 'Minutes_Lightly_Active',
    'Sedentary': FITBIT_DIR / 'Minutes_Sedentary',
    'Very_Active': FITBIT_DIR / 'Minutes_Very_Active',
}

print(f"\nValidating Fitbit data sources:")
sources_found = 0
for name, path in fitbit_sources.items():
    exists = path.exists()
    if exists:
        files = list(path.glob('*.csv'))
        print(f"  ✓ {name:20s}: {len(files):3d} files")
        sources_found += 1
    else:
        print(f"  ✗ {name:20s}: NOT FOUND")

if sources_found == 0:
    raise FileNotFoundError(f"No Fitbit data sources found at {FITBIT_DIR}")

# Get all participant IDs from Steps directory (base timeline)
steps_dir = FITBIT_DIR / 'Steps'
if not steps_dir.exists() or len(list(steps_dir.glob('*.csv'))) == 0:
    raise FileNotFoundError(f"No Step files found in {steps_dir}")

step_files = sorted(steps_dir.glob('*.csv'))
pids = [clean_pid(p.stem) for p in step_files]
print(f"\nFound {len(pids)} unique participants")

# Build per-participant Fitbit dataset
print(f"\nProcessing minute-level Fitbit data for {len(pids)} participants...")

df_list = []
failed_count = 0

for idx, pid in enumerate(pids, 1):
    if idx % max(1, len(pids) // 10) == 0 or idx == len(pids):
        print(f"  Progress: {idx}/{len(pids)} participants...")
    
    try:
        # Load Steps as base timeline
        steps = load_fitbit_csv(steps_dir / f"{pid}.csv", 'Steps')
        
        if len(steps) == 0:
            failed_count += 1
            continue
        
        # Use Steps as baseline
        merged = steps[['pid', 'dateTime']].drop_duplicates().sort_values('dateTime')
        merged = merged.merge(steps, on=['pid', 'dateTime'], how='left')
        
        # Merge other Fitbit data sources
        for source_name, source_dir in fitbit_sources.items():
            if source_name == 'Steps':
                continue
            
            source_file = source_dir / f"{pid}.csv"
            source_data = load_fitbit_csv(source_file, source_name)
            
            if len(source_data) > 0:
                merged = merged.merge(source_data, on=['pid', 'dateTime'], how='left')
        
        df_list.append(merged)
    
    except Exception as e:
        failed_count += 1
        continue

print(f"\nConcatenating data from {len(df_list)} successful participants...")
if len(df_list) == 0:
    raise ValueError(f"No valid Fitbit data loaded. Failed: {failed_count} participants")

df = pd.concat(df_list, ignore_index=True)
df = df.sort_values(['pid', 'dateTime']).reset_index(drop=True)

print(f"\n✓ Fitbit dataset successfully created:")
print(f"  • Total rows:        {len(df):,}")
print(f"  • Total columns:     {len(df.columns)}")
print(f"  • Participants:      {df['pid'].nunique()}")
print(f"  • Date range:        {df['dateTime'].min().date()} to {df['dateTime'].max().date()}")
print(f"  • Duration:          {(df['dateTime'].max() - df['dateTime'].min()).days} days")
print(f"  • Skipped:           {failed_count} participants")



STEP 1: LOADING FITBIT SENSOR DATA AT MINUTE RESOLUTION

Validating Fitbit data sources:
  ✓ Steps               : 166 files
  ✓ Floors              : 166 files
  ✓ HR                  : 161 files
  ✓ Fairly_Active       : 166 files
  ✓ Lightly_Active      : 166 files
  ✓ Sedentary           : 166 files
  ✓ Very_Active         : 166 files

Found 166 unique participants

Processing minute-level Fitbit data for 166 participants...
  Progress: 16/166 participants...
  Progress: 32/166 participants...
  Progress: 48/166 participants...
  Progress: 64/166 participants...
  Progress: 80/166 participants...
  Progress: 96/166 participants...
  Progress: 112/166 participants...
  Progress: 128/166 participants...
  Progress: 144/166 participants...
  Progress: 160/166 participants...
  Progress: 166/166 participants...

Concatenating data from 166 successful participants...

✓ Fitbit dataset successfully created:
  • Total rows:        53,104,440
  • Total columns:     9
  • Participants:    

## Step 4: Process and Merge Sleep Data at Minute Resolution

In [4]:
print("\n" + "="*100)
print("STEP 2: LOADING AND EXPANDING SLEEP DATA TO MINUTE LEVEL")
print("="*100)

sleep_dir = FITBIT_DIR / 'Sleep_Intraday'

if not sleep_dir.exists():
    print(f"\n✗ Sleep directory not found at {sleep_dir}")
    print("   Adding empty sleep_level column")
    df['sleep_level'] = pd.NA
else:
    sleep_files = list(sleep_dir.glob('*.csv'))
    print(f"\nFound {len(sleep_files)} sleep files")
    
    if len(sleep_files) == 0:
        print("   No sleep files found")
        df['sleep_level'] = pd.NA
    else:
        sleep_records = []
        
        for sleep_file in sleep_files:
            pid = clean_pid(sleep_file.stem)
            
            try:
                sleep_df = pd.read_csv(sleep_file)
                
                # Identify columns dynamically
                time_col = next((c for c in sleep_df.columns if 'date' in c.lower() or 'time' in c.lower()), None)
                level_col = next((c for c in sleep_df.columns if 'level' in c.lower()), None)
                duration_col = next((c for c in sleep_df.columns if 'duration' in c.lower()), None)
                
                if not all([time_col, level_col, duration_col]):
                    continue
                
                # Clean datetime and duration
                sleep_df['local_date_time'] = pd.to_datetime(sleep_df[time_col], errors='coerce')
                sleep_df['duration_sec'] = pd.to_numeric(sleep_df[duration_col], errors='coerce')
                sleep_df['level'] = sleep_df[level_col].astype(str)
                
                # Filter valid rows
                valid_rows = sleep_df[sleep_df['local_date_time'].notna() & sleep_df['duration_sec'].notna()]
                
                if len(valid_rows) == 0:
                    continue
                
                # Vectorized expansion: create minute ranges
                for _, row in valid_rows.iterrows():
                    start_time = pd.Timestamp(row['local_date_time']).floor('min')
                    duration_min = max(1, int(row['duration_sec'] / 60))
                    
                    # Generate minute timestamps
                    minute_times = pd.date_range(start=start_time, periods=duration_min, freq='min')
                    
                    for minute_time in minute_times:
                        sleep_records.append({
                            'pid': pid,
                            'dateTime': minute_time,
                            'sleep_level': row['level']
                        })
            
            except Exception as e:
                continue
        
        if len(sleep_records) > 0:
            sleep_data = pd.DataFrame(sleep_records)
            
            # Remove duplicates (keep most recent if any conflicts)
            sleep_data = sleep_data.sort_values(['pid', 'dateTime']).drop_duplicates(
                subset=['pid', 'dateTime'], 
                keep='last'
            )
            
            print(f"\n✓ Sleep data expanded: {len(sleep_data):,} minute-level records")
            
            # Merge with main dataframe
            df['dateTime'] = pd.to_datetime(df['dateTime'])
            df = df.merge(sleep_data, on=['pid', 'dateTime'], how='left')
            print(f"  Dataset after merge: {len(df):,} rows")
        else:
            print("   No valid sleep records found")
            df['sleep_level'] = pd.NA

print(f"\n✓ Total dataset rows: {len(df):,}")



STEP 2: LOADING AND EXPANDING SLEEP DATA TO MINUTE LEVEL

Found 161 sleep files

✓ Sleep data expanded: 14,571,543 minute-level records
  Dataset after merge: 53,104,440 rows

✓ Total dataset rows: 53,104,440


## Step 5: Load and Merge AWARE Screen Events

In [5]:
print("\n" + "="*100)
print("STEP 3: LOADING AND MERGING AWARE SCREEN EVENTS")
print("="*100)

screen_dir = AWARE_DIR / 'Screen'

if not screen_dir.exists():
    print(f"\n✗ Screen directory not found at {screen_dir}")
    df['screen'] = pd.NA
else:
    screen_files = list(screen_dir.glob('*.csv'))
    print(f"\nFound {len(screen_files)} screen event files")
    
    if len(screen_files) == 0:
        print("   No screen files found")
        df['screen'] = pd.NA
    else:
        screen_records = []
        failed_count = 0
        
        for screen_file in screen_files:
            pid = clean_pid(screen_file.stem)
            
            try:
                screen_df = pd.read_csv(screen_file)
                
                # Identify columns dynamically
                time_col = next((c for c in screen_df.columns if 'date' in c.lower() or 'time' in c.lower()), None)
                screen_col = next((c for c in screen_df.columns if 'screen' in c.lower()), None)
                
                if not time_col or not screen_col:
                    failed_count += 1
                    continue
                
                # Clean data
                screen_df['event_time'] = pd.to_datetime(screen_df[time_col], errors='coerce')
                screen_df['screen'] = pd.to_numeric(screen_df[screen_col], errors='coerce')
                screen_df['pid'] = pid
                
                # Keep only valid rows
                valid_df = screen_df[['pid', 'event_time', 'screen']].dropna(subset=['event_time'])
                
                if len(valid_df) > 0:
                    screen_records.append(valid_df)
                else:
                    failed_count += 1
            
            except Exception as e:
                failed_count += 1
                continue
        
        if len(screen_records) > 0:
            screen_data = pd.concat(screen_records, ignore_index=True)
            print(f"\nLoaded {len(screen_data):,} screen events from {len(screen_records)} participants")
            
            # Initialize screen column
            df['screen'] = np.nan
            
            # Merge using merge_asof per participant
            print("   Merging screen events with main dataset...")
            
            df['dateTime'] = pd.to_datetime(df['dateTime'])
            
            for i, pid_val in enumerate(screen_data['pid'].unique()):
                if (i + 1) % 20 == 0:
                    print(f"  • Processing participant {i+1}/{len(screen_data['pid'].unique())}...")
                
                df_pid = df[df['pid'] == pid_val][['pid', 'dateTime']].copy()
                
                if len(df_pid) == 0:
                    continue
                
                pid_screen = screen_data[screen_data['pid'] == pid_val].copy()
                pid_screen = pid_screen.sort_values('event_time').rename(
                    columns={'event_time': 'dateTime'}
                )[['dateTime', 'screen']].copy()
                
                # Ensure datetime format consistency
                df_pid['dateTime'] = pd.to_datetime(df_pid['dateTime'], utc=False).astype('datetime64[ns]')
                pid_screen['dateTime'] = pd.to_datetime(pid_screen['dateTime'], utc=False).astype('datetime64[ns]')
                
                # Use merge_asof: each minute gets the most recent screen state
                pid_merged = pd.merge_asof(
                    df_pid.sort_values('dateTime'),
                    pid_screen.sort_values('dateTime'),
                    on='dateTime',
                    direction='backward'
                )
                
                # Update screen column for this participant
                df.loc[df['pid'] == pid_val, 'screen'] = pid_merged['screen'].values
            
            print(f"\n✓ Screen data merged: {len(df):,} rows")
        else:
            print(f"   No valid screen records found (failed: {failed_count})")
            df['screen'] = pd.NA

print(f"\n✓ Total dataset rows: {len(df):,}")



STEP 3: LOADING AND MERGING AWARE SCREEN EVENTS

Found 165 screen event files

Loaded 6,989,103 screen events from 165 participants
   Merging screen events with main dataset...
  • Processing participant 20/165...
  • Processing participant 40/165...
  • Processing participant 60/165...
  • Processing participant 80/165...
  • Processing participant 100/165...
  • Processing participant 120/165...
  • Processing participant 140/165...
  • Processing participant 160/165...

✓ Screen data merged: 53,104,440 rows

✓ Total dataset rows: 53,104,440


## Step 7: Calculate Depression Labels from CES-D Surveys

In [6]:
print("\n" + "="*100)
print("STEP 5: CALCULATE DEPRESSION LABELS FROM CES-D SURVEYS")
print("="*100)

# ═══════════════════════════════════════════════════════════════════════════════
# CES-D (Center for Epidemiological Studies - Depression) Scale
# ═══════════════════════════════════════════════════════════════════════════════
# Reference: Radloff, L. S. (1977). The CES-D Scale: A self-report depression 
# scale for research in the general population. Applied Psychological Measurement.
#
# SCORING DETAILS:
#   • 20 items total
#   • 4-point Likert scale per item: 0-3 (rarely → most of the time)
#   • Total range: 0-60
#   • Depression threshold: CES-D ≥ 16
#   • Reverse-scored items (4): Q4, Q8, Q12, Q16 (positive affect)
#     These measure GOOD mood, so high scores should mean HEALTHY
#
# REVERSE SCORING:
#   Original  →  Reversed
#   0 (rarely) → 3 (most)
#   1 (little) → 2 (moderate)
#   2 (moderate) → 1 (little)
#   3 (most) → 0 (rarely)

# Response mapping: frequency of depressive symptoms
ces_d_mapping = {
    'Rarely or none of the time (less than 1 day)': 0,
    'Some or a little of the time (1-2 days)': 1,
    'Occasionally or a moderate amount of time (3-4 days)': 2,
    'Most or all of the time (5-7 days)': 3
}

# Keywords for reverse-scored items (Q4, Q8, Q12, Q16 - positive affect items)
# These items measure GOOD feelings, so reverse scoring is needed
reverse_scored_keywords = ['good', 'hopeful', 'happy', 'enjoyed', 'life']

def score_ces_d(row):
    """
    Calculate CES-D total score (0-60 range).
    
    Algorithm:
    1. Extract all response columns (exclude PID and metadata)
    2. For each item:
       - Map response text to numeric (0-3)
       - Check if item is reverse-scored (positive affect items)
       - If reverse: score = 3 - original_score
    3. Sum all item scores
    4. Return score only if ≥10 items present (50% minimum)
    
    Returns:
        CES-D total score (0-60), or NaN if insufficient data
    """
    total_score = 0
    valid_items = 0
    
    for col in row.index:
        # Skip non-question columns
        if col == 'PID' or col.startswith('_'):
            continue
        
        val = row[col]
        if pd.isna(val) or val == '':
            continue
        
        # 1. Map text response to numeric (0-3)
        numeric_val = ces_d_mapping.get(str(val).strip(), None)
        if numeric_val is None:
            continue
        
        # 2. Check if this is a reverse-scored item (positive affect)
        is_reverse = any(keyword in str(col).lower() for keyword in reverse_scored_keywords)
        
        # 3. Apply reverse scoring if needed
        if is_reverse:
            numeric_val = 3 - numeric_val  # Reverse: 0↔3, 1↔2
        
        total_score += numeric_val
        valid_items += 1
    
    # 4. Return score only if at least half of 20 items are present (10+ items)
    return total_score if valid_items >= 10 else np.nan

# Initialize depression tracking dictionary
depression_status = {}

# Load Pre-Study CES-D
pre_cesd_file = SURVEY_DIR / 'Pre-Study_Baselines' / 'CES-D.csv'
pre_cesd = None

if pre_cesd_file.exists():
    try:
        print("\nLoading Pre-Study CES-D (Baseline Assessment)...")
        pre_cesd = pd.read_csv(pre_cesd_file)
        
        # Calculate CES-D scores
        pre_cesd['CES_D_Total'] = pre_cesd.apply(score_ces_d, axis=1)
        pre_cesd = pre_cesd.dropna(subset=['CES_D_Total'])
        
        # Classify depression status: threshold ≥ 16
        pre_cesd['depression_status_baseline'] = (pre_cesd['CES_D_Total'] >= 16).astype(int)
        
        # Build depression status dictionary
        for _, row in pre_cesd.iterrows():
            pid = clean_pid(row['PID'])
            depression_status[pid] = {
                'ces_d_baseline_score': row['CES_D_Total'],
                'depression_status_baseline': row['depression_status_baseline']
            }
        
        print(f"✓ Pre-Study CES-D: {len(pre_cesd)} participants scored")
        healthy_count = (pre_cesd['CES_D_Total'] < 16).sum()
        depressed_count = (pre_cesd['CES_D_Total'] >= 16).sum()
        print(f"  • Healthy (CES-D < 16):   {healthy_count} ({healthy_count/len(pre_cesd)*100:.1f}%)")
        print(f"  • Depressed (CES-D ≥ 16): {depressed_count} ({depressed_count/len(pre_cesd)*100:.1f}%)")
        print(f"  • Mean ± SD:              {pre_cesd['CES_D_Total'].mean():.2f} ± {pre_cesd['CES_D_Total'].std():.2f}")
    except Exception as e:
        print(f"✗ Error loading Pre-Study CES-D: {e}")
else:
    print(f"\n✗ Pre-Study CES-D not found at {pre_cesd_file}")

# Load Post-Study CES-D
post_cesd_file = SURVEY_DIR / 'Post-Study_Baselines' / 'CES-D.csv'
post_cesd = None

if post_cesd_file.exists():
    try:
        print("\nLoading Post-Study CES-D (Endpoint Assessment)...")
        post_cesd = pd.read_csv(post_cesd_file)
        
        # Calculate CES-D scores
        post_cesd['CES_D_Total'] = post_cesd.apply(score_ces_d, axis=1)
        post_cesd = post_cesd.dropna(subset=['CES_D_Total'])
        
        # Classify depression status: threshold ≥ 16
        post_cesd['depression_status_endpoint'] = (post_cesd['CES_D_Total'] >= 16).astype(int)
        
        # Update depression status dictionary
        for _, row in post_cesd.iterrows():
            pid = clean_pid(row['PID'])
            if pid in depression_status:
                depression_status[pid]['ces_d_endpoint_score'] = row['CES_D_Total']
                depression_status[pid]['depression_status_endpoint'] = row['depression_status_endpoint']
            else:
                depression_status[pid] = {
                    'ces_d_endpoint_score': row['CES_D_Total'],
                    'depression_status_endpoint': row['depression_status_endpoint']
                }
        
        print(f"✓ Post-Study CES-D: {len(post_cesd)} participants scored")
        healthy_count = (post_cesd['CES_D_Total'] < 16).sum()
        depressed_count = (post_cesd['CES_D_Total'] >= 16).sum()
        print(f"  • Healthy (CES-D < 16):   {healthy_count} ({healthy_count/len(post_cesd)*100:.1f}%)")
        print(f"  • Depressed (CES-D ≥ 16): {depressed_count} ({depressed_count/len(post_cesd)*100:.1f}%)")
        print(f"  • Mean ± SD:              {post_cesd['CES_D_Total'].mean():.2f} ± {post_cesd['CES_D_Total'].std():.2f}")
    except Exception as e:
        print(f"✗ Error loading Post-Study CES-D: {e}")
else:
    print(f"\n✗ Post-Study CES-D not found at {post_cesd_file}")

# ═══════════════════════════════════════════════════════════════════════════════
# CLASSIFY DEPRESSION TRAJECTORIES (Longitudinal Categories)
# ═══════════════════════════════════════════════════════════════════════════════
# Based on Pre→Post depression status transitions:
#   0→0: Stable_Healthy (remained healthy throughout study)
#   0→1: Onset (developed depression during study)
#   1→0: Recovery (remitted from depression during study)
#   1→1: Persistent (depressed at both timepoints)
#   Mixed: Unknown (incomplete baseline or endpoint data)

print("\nClassifying depression trajectories (Pre → Post)...")

trajectory_counts = {'Stable_Healthy': 0, 'Onset': 0, 'Recovery': 0, 'Persistent': 0, 'Unknown': 0}

for pid in depression_status:
    baseline = depression_status[pid].get('depression_status_baseline', -1)
    endpoint = depression_status[pid].get('depression_status_endpoint', -1)
    
    if baseline == -1 or endpoint == -1:
        depression_status[pid]['depression_trajectory'] = 'Unknown'
        trajectory_counts['Unknown'] += 1
    elif baseline == 0 and endpoint == 0:
        depression_status[pid]['depression_trajectory'] = 'Stable_Healthy'
        trajectory_counts['Stable_Healthy'] += 1
    elif baseline == 0 and endpoint == 1:
        depression_status[pid]['depression_trajectory'] = 'Onset'
        trajectory_counts['Onset'] += 1
    elif baseline == 1 and endpoint == 0:
        depression_status[pid]['depression_trajectory'] = 'Recovery'
        trajectory_counts['Recovery'] += 1
    elif baseline == 1 and endpoint == 1:
        depression_status[pid]['depression_trajectory'] = 'Persistent'
        trajectory_counts['Persistent'] += 1

print(f"\n✓ Depression trajectories classified ({len(depression_status)} participants):")
for trajectory, count in trajectory_counts.items():
    if count > 0:
        print(f"  • {trajectory:20s}: {count:4d} participants")


STEP 5: CALCULATE DEPRESSION LABELS FROM CES-D SURVEYS

Loading Pre-Study CES-D (Baseline Assessment)...
✓ Pre-Study CES-D: 163 participants scored
  • Healthy (CES-D < 16):   68 (41.7%)
  • Depressed (CES-D ≥ 16): 95 (58.3%)
  • Mean ± SD:              19.97 ± 10.95

Loading Post-Study CES-D (Endpoint Assessment)...
✓ Post-Study CES-D: 118 participants scored
  • Healthy (CES-D < 16):   51 (43.2%)
  • Depressed (CES-D ≥ 16): 67 (56.8%)
  • Mean ± SD:              19.77 ± 10.78

Classifying depression trajectories (Pre → Post)...

✓ Depression trajectories classified (162 participants):
  • Stable_Healthy      :   37 participants
  • Onset               :   14 participants
  • Recovery            :   13 participants
  • Persistent          :   53 participants
  • Unknown             :   45 participants


In [7]:
print("\n" + "="*100)
print("STEP 4: LOADING AND MERGING AWARE CALL DATA")
print("="*100)

calls_dir = AWARE_DIR / 'Calls'

if not calls_dir.exists():
    print(f"\n✗ Calls directory not found at {calls_dir}")
    df['calls'] = pd.NA
else:
    call_files = list(calls_dir.glob('*.csv'))
    print(f"\nFound {len(call_files)} call event files")
    
    if len(call_files) == 0:
        print("   No call files found")
        df['calls'] = pd.NA
    else:
        call_records = []
        failed_count = 0
        
        for call_file in call_files:
            pid = clean_pid(call_file.stem)
            
            try:
                calls_df = pd.read_csv(call_file)
                
                # Identify columns dynamically
                time_col = next((c for c in calls_df.columns if 'date' in c.lower() or 'time' in c.lower()), None)
                call_col = next((c for c in calls_df.columns if 'call' in c.lower()), None)
                
                # If no call column, check for state or type
                if not call_col:
                    call_col = next((c for c in calls_df.columns if 'state' in c.lower() or 'type' in c.lower()), None)
                
                if not time_col or not call_col:
                    failed_count += 1
                    continue
                
                # Clean data
                calls_df['event_time'] = pd.to_datetime(calls_df[time_col], errors='coerce')
                calls_df['calls'] = pd.to_numeric(calls_df[call_col], errors='coerce')
                calls_df['pid'] = pid
                
                # Keep only valid rows
                valid_df = calls_df[['pid', 'event_time', 'calls']].dropna(subset=['event_time'])
                
                if len(valid_df) > 0:
                    call_records.append(valid_df)
                else:
                    failed_count += 1
            
            except Exception as e:
                failed_count += 1
                continue
        
        if len(call_records) > 0:
            call_data = pd.concat(call_records, ignore_index=True)
            print(f"\nLoaded {len(call_data):,} call events from {len(call_records)} participants")
            
            # Initialize calls column
            df['calls'] = np.nan
            
            # Merge using merge_asof per participant
            print("   Merging call events with main dataset...")
            
            df['dateTime'] = pd.to_datetime(df['dateTime'])
            
            for i, pid_val in enumerate(call_data['pid'].unique()):
                if (i + 1) % 20 == 0:
                    print(f"  • Processing participant {i+1}/{len(call_data['pid'].unique())}...")
                
                df_pid = df[df['pid'] == pid_val][['pid', 'dateTime']].copy()
                
                if len(df_pid) == 0:
                    continue
                
                pid_calls = call_data[call_data['pid'] == pid_val].copy()
                pid_calls = pid_calls.sort_values('event_time').rename(
                    columns={'event_time': 'dateTime'}
                )[['dateTime', 'calls']].copy()
                
                # Ensure datetime format consistency
                df_pid['dateTime'] = pd.to_datetime(df_pid['dateTime'], utc=False).astype('datetime64[ns]')
                pid_calls['dateTime'] = pd.to_datetime(pid_calls['dateTime'], utc=False).astype('datetime64[ns]')
                
                # Use merge_asof: each minute gets the most recent call state
                pid_merged = pd.merge_asof(
                    df_pid.sort_values('dateTime'),
                    pid_calls.sort_values('dateTime'),
                    on='dateTime',
                    direction='backward'
                )
                
                # Update calls column for this participant
                df.loc[df['pid'] == pid_val, 'calls'] = pid_merged['calls'].values
            
            print(f"\n✓ Call data merged: {len(df):,} rows")
        else:
            print(f"   No valid call records found (failed: {failed_count})")
            df['calls'] = pd.NA

print(f"\n✓ Total dataset rows: {len(df):,}")


STEP 4: LOADING AND MERGING AWARE CALL DATA

Found 155 call event files

Loaded 373,245 call events from 155 participants
   Merging call events with main dataset...
  • Processing participant 20/155...
  • Processing participant 40/155...
  • Processing participant 60/155...
  • Processing participant 80/155...
  • Processing participant 100/155...
  • Processing participant 120/155...
  • Processing participant 140/155...

✓ Call data merged: 53,104,440 rows

✓ Total dataset rows: 53,104,440


## Step 6: Check AWARE Call Data (Optional)

In [8]:
print("\nMapping depression labels to minute-level records...")

# Define expected columns at this point (before mapping depression labels)
expected_cols = ['pid', 'dateTime', 'Steps', 'Floors', 'HR', 'Fairly_Active', 
                 'Lightly_Active', 'Sedentary', 'Very_Active', 'sleep_level', 'screen', 'calls']

# Remove any unexpected columns (including survey question columns)
unexpected_cols = [col for col in df.columns if col not in expected_cols]
if unexpected_cols:
    print(f"  ⚠ Removing unexpected columns: {unexpected_cols}")
    df = df.drop(columns=unexpected_cols)

# Create lookup dictionaries from depression_status
depression_baseline_map = {pid: info.get('depression_status_baseline', np.nan) 
                           for pid, info in depression_status.items()}
depression_endpoint_map = {pid: info.get('depression_status_endpoint', np.nan) 
                          for pid, info in depression_status.items()}
trajectory_map = {pid: info.get('depression_trajectory', np.nan) 
                 for pid, info in depression_status.items()}
cesd_baseline_map = {pid: info.get('ces_d_baseline_score', np.nan) 
                    for pid, info in depression_status.items()}
cesd_endpoint_map = {pid: info.get('ces_d_endpoint_score', np.nan) 
                    for pid, info in depression_status.items()}

# Process in chunks for memory efficiency (large datasets)
chunk_size = 100000
num_chunks = max(1, (len(df) + chunk_size - 1) // chunk_size)

print(f"Processing {len(df):,} rows in {num_chunks} chunk(s)...")

# Initialize new columns with better names
df['depression_status_baseline'] = np.nan
df['depression_status_endpoint'] = np.nan
df['depression_trajectory'] = np.nan
df['ces_d_baseline_score'] = np.nan
df['ces_d_endpoint_score'] = np.nan

# Apply mapping to each chunk
for chunk_num, i in enumerate(range(0, len(df), chunk_size), 1):
    end_idx = min(i + chunk_size, len(df))
    
    # Progress indicator
    if num_chunks > 1 and chunk_num % max(1, num_chunks // 5) == 0:
        print(f"  • Chunk {chunk_num}/{num_chunks}...")
    
    # Map depression labels for this chunk
    chunk_pids = df.loc[i:end_idx-1, 'pid']
    
    df.loc[i:end_idx-1, 'depression_status_baseline'] = chunk_pids.map(depression_baseline_map)
    df.loc[i:end_idx-1, 'depression_status_endpoint'] = chunk_pids.map(depression_endpoint_map)
    df.loc[i:end_idx-1, 'depression_trajectory'] = chunk_pids.map(trajectory_map)
    df.loc[i:end_idx-1, 'ces_d_baseline_score'] = chunk_pids.map(cesd_baseline_map)
    df.loc[i:end_idx-1, 'ces_d_endpoint_score'] = chunk_pids.map(cesd_endpoint_map)

print(f"✓ Depression labels successfully added")

# Verify coverage
baseline_count = df['depression_status_baseline'].notna().sum()
endpoint_count = df['depression_status_endpoint'].notna().sum()
trajectory_count = df['depression_trajectory'].notna().sum()

print(f"  • Baseline depression status:  {baseline_count:,} records ({baseline_count/len(df)*100:.1f}%)")
print(f"  • Endpoint depression status:  {endpoint_count:,} records ({endpoint_count/len(df)*100:.1f}%)")
print(f"  • Depression trajectory:       {trajectory_count:,} records ({trajectory_count/len(df)*100:.1f}%)")


Mapping depression labels to minute-level records...
Processing 53,104,440 rows in 532 chunk(s)...
  • Chunk 106/532...
  • Chunk 212/532...
  • Chunk 318/532...
  • Chunk 424/532...
  • Chunk 530/532...
✓ Depression labels successfully added
  • Baseline depression status:  51,962,700 records (97.9%)
  • Endpoint depression status:  37,561,560 records (70.7%)
  • Depression trajectory:       52,296,720 records (98.5%)


## Step 8: Map Depression Labels to Main Dataset

In [9]:
print("\n" + "="*100)
print("DEPRESSION LABEL ANALYSIS")
print("="*100)

# Get unique participant labels
pid_labels = df[['pid', 'depression_status_baseline', 'depression_status_endpoint', 'depression_trajectory']].drop_duplicates('pid')

print(f"\n1. BASELINE DEPRESSION STATUS (CES-D ≥ 16)")
print(f"   {'='*80}")
baseline_counts = pid_labels['depression_status_baseline'].value_counts(dropna=False)
print(f"   Healthy (0):        {baseline_counts.get(0, 0):4d} participants")
print(f"   Depressed (1):      {baseline_counts.get(1, 0):4d} participants")
print(f"   Missing:            {baseline_counts.get(np.nan, 0):4d} participants")
print(f"   Total labeled:      {baseline_counts.sum():4d} participants")

print(f"\n2. ENDPOINT DEPRESSION STATUS (CES-D ≥ 16)")
print(f"   {'='*80}")
endpoint_counts = pid_labels['depression_status_endpoint'].value_counts(dropna=False)
print(f"   Healthy (0):        {endpoint_counts.get(0, 0):4d} participants")
print(f"   Depressed (1):      {endpoint_counts.get(1, 0):4d} participants")
print(f"   Missing:            {endpoint_counts.get(np.nan, 0):4d} participants")
print(f"   Total labeled:      {endpoint_counts.sum():4d} participants")

print(f"\n3. DEPRESSION TRAJECTORY (Pre → Post Transition)")
print(f"   {'='*80}")
trajectory_counts = pid_labels['depression_trajectory'].value_counts(dropna=False)
for cat, count in trajectory_counts.items():
    if pd.isna(cat):
        print(f"   Unknown:            {count:4d} participants")
    else:
        print(f"   {cat:20s}: {count:4d} participants")

print(f"\n4. CES-D SCORE STATISTICS")
print(f"   {'='*80}")
print(f"\n   Baseline CES-D Scores (0-60, threshold=16):")
print(f"   - Mean ± SD:   {df['ces_d_baseline_score'].mean():6.2f} ± {df['ces_d_baseline_score'].std():.2f}")
print(f"   - Median:      {df['ces_d_baseline_score'].median():6.2f}")
print(f"   - Min - Max:   {df['ces_d_baseline_score'].min():6.2f} - {df['ces_d_baseline_score'].max():.2f}")

print(f"\n   Endpoint CES-D Scores (0-60, threshold=16):")
print(f"   - Mean ± SD:   {df['ces_d_endpoint_score'].mean():6.2f} ± {df['ces_d_endpoint_score'].std():.2f}")
print(f"   - Median:      {df['ces_d_endpoint_score'].median():6.2f}")
print(f"   - Min - Max:   {df['ces_d_endpoint_score'].min():6.2f} - {df['ces_d_endpoint_score'].max():.2f}")

print(f"\n✓ Label analysis complete")


DEPRESSION LABEL ANALYSIS

1. BASELINE DEPRESSION STATUS (CES-D ≥ 16)
   Healthy (0):          68 participants
   Depressed (1):        93 participants
   Missing:               5 participants
   Total labeled:       166 participants

2. ENDPOINT DEPRESSION STATUS (CES-D ≥ 16)
   Healthy (0):          51 participants
   Depressed (1):        67 participants
   Missing:              48 participants
   Total labeled:       166 participants

3. DEPRESSION TRAJECTORY (Pre → Post Transition)
   Persistent          :   53 participants
   Unknown             :   45 participants
   Stable_Healthy      :   37 participants
   Onset               :   14 participants
   Recovery            :   13 participants
   Unknown:               4 participants

4. CES-D SCORE STATISTICS

   Baseline CES-D Scores (0-60, threshold=16):
   - Mean ± SD:    19.37 ± 10.77
   - Median:       17.00
   - Min - Max:     4.00 - 53.00

   Endpoint CES-D Scores (0-60, threshold=16):
   - Mean ± SD:    19.65 ± 10.87
   -

## Step 10: Analyze Depression Labels Distribution

In [10]:
print("\n" + "="*100)
print("FINALIZING AND SAVING DATASET")
print("="*100)

# Remove any duplicate columns
df = df.loc[:, ~df.columns.duplicated()]

# Remove survey question columns (Q3-Q7) that should not be in final dataset
survey_question_cols = [
    'Q3_Cognitive_Energy', 'Q4_Physical_Energy', 'Q5_Emotional_Energy',
    'Q6_5_Professional_Activities', 'Q6_6_Social_Activities', 
    'Q6_7_Physical_Activities', 'Q6_8_Leisure_Activities', 'Q7_Mental_Demand'
]

cols_to_drop = [col for col in survey_question_cols if col in df.columns]
if cols_to_drop:
    print(f"\n✓ Removing survey question columns: {cols_to_drop}")
    df = df.drop(columns=cols_to_drop)

print(f"\nDataset ready for save:")
print(f"  Rows: {len(df):,}")
print(f"  Columns: {len(df.columns)}")
print(f"  Participants: {df['pid'].nunique()}")
print(f"  Date range: {df['dateTime'].min()} to {df['dateTime'].max()}")
print(f"  Duration: {(df['dateTime'].max() - df['dateTime'].min()).days} days")

print(f"\nSaving to {OUTPUT_FILE.name}...")

try:
    df.to_csv(OUTPUT_FILE, index=False)
    file_size = OUTPUT_FILE.stat().st_size / (1024**3)
    print(f"\n✓ Successfully saved!")
    print(f"  File: {OUTPUT_FILE}")
    print(f"  Size: {file_size:.3f} GB")
except Exception as e:
    print(f"\n✗ Error saving: {e}")

print(f"\n✓ Dataset structure ({len(df.columns)} columns):")
for i, col in enumerate(df.columns, 1):
    dtype = str(df[col].dtype)
    non_null = df[col].notna().sum()
    pct = non_null / len(df) * 100
    print(f"  {i:2d}. {col:30s} {dtype:12s} {non_null:10,} records ({pct:5.1f}%)")


FINALIZING AND SAVING DATASET

Dataset ready for save:
  Rows: 53,104,440
  Columns: 17
  Participants: 166
  Date range: 2021-09-07 00:00:00 to 2023-01-17 23:59:00
  Duration: 497 days

Saving to HRD_RAW_MinuteLevel.csv...

✓ Successfully saved!
  File: c:\Users\umroot\Desktop\Human-Rhythms-Dataset\HRD\HRD_RAW_MinuteLevel.csv
  Size: 4.025 GB

✓ Dataset structure (17 columns):
   1. pid                            object       53,104,440 records (100.0%)
   2. dateTime                       datetime64[ns] 53,104,440 records (100.0%)
   3. Steps                          int64        53,104,440 records (100.0%)
   4. Floors                         int64        53,104,440 records (100.0%)
   5. HR                             float64      43,929,564 records ( 82.7%)
   6. Fairly_Active                  float64      53,104,432 records (100.0%)
   7. Lightly_Active                 float64      53,104,432 records (100.0%)
   8. Sedentary                      int64        53,104,440 records (

## Step 9: Save Complete Merged Dataset

## Step 11: Analyze Data Completeness Across All Streams

In [11]:
print("\n" + "="*100)
print("MISSING DATA ANALYSIS")
print("="*100)

print(f"\n1. FITBIT SENSOR DATA COMPLETENESS")
print(f"   {'='*80}")
sensor_cols = ['HR', 'Steps', 'Floors', 'Fairly_Active', 'Lightly_Active', 'Sedentary', 'Very_Active']
for col in sensor_cols:
    if col in df.columns:
        non_null = df[col].notna().sum()
        pct = (non_null / len(df)) * 100
        print(f"   {col:20s}: {pct:6.2f}% complete ({non_null:,} of {len(df):,} records)")

print(f"\n2. OTHER SENSOR DATA COMPLETENESS")
print(f"   {'='*80}")
other_cols = ['sleep_level', 'screen', 'Calls']
for col in other_cols:
    if col in df.columns:
        non_null = df[col].notna().sum()
        pct = (non_null / len(df)) * 100
        print(f"   {col:20s}: {pct:6.2f}% complete ({non_null:,} of {len(df):,} records)")

print(f"\n3. DEPRESSION LABEL COMPLETENESS")
print(f"   {'='*80}")
label_cols = ['depression_status_baseline', 'depression_status_endpoint', 'depression_trajectory', 'ces_d_baseline_score', 'ces_d_endpoint_score']
for col in label_cols:
    if col in df.columns:
        non_null = df[col].notna().sum()
        pct = (non_null / len(df)) * 100
        print(f"   {col:35s}: {pct:6.2f}% ({non_null:,} records)")

print(f"\n4. DATA RECOVERY RATE")
print(f"   {'='*80}")
expected_per_day = 24 * 60  # 1440 minutes
study_info = df.groupby('pid').agg({'dateTime': ['min', 'max', 'count']})
study_info.columns = ['start', 'end', 'actual_count']
study_info['duration_days'] = (study_info['end'] - study_info['start']).dt.days + 1
study_info['expected_count'] = study_info['duration_days'] * expected_per_day
study_info['recovery_rate'] = (study_info['actual_count'] / study_info['expected_count'] * 100).fillna(0)

print(f"   Overall data recovery: {(len(df) / study_info['expected_count'].sum() * 100):.1f}%")
print(f"   Mean recovery per participant: {study_info['recovery_rate'].mean():.1f}%")
print(f"   Median recovery per participant: {study_info['recovery_rate'].median():.1f}%")
print(f"   Min recovery: {study_info['recovery_rate'].min():.1f}%")
print(f"   Max recovery: {study_info['recovery_rate'].max():.1f}%")

print(f"\n✓ Missing data analysis complete")


MISSING DATA ANALYSIS

1. FITBIT SENSOR DATA COMPLETENESS
   HR                  :  82.72% complete (43,929,564 of 53,104,440 records)
   Steps               : 100.00% complete (53,104,440 of 53,104,440 records)
   Floors              : 100.00% complete (53,104,440 of 53,104,440 records)
   Fairly_Active       : 100.00% complete (53,104,432 of 53,104,440 records)
   Lightly_Active      : 100.00% complete (53,104,432 of 53,104,440 records)
   Sedentary           : 100.00% complete (53,104,440 of 53,104,440 records)
   Very_Active         : 100.00% complete (53,104,440 of 53,104,440 records)

2. OTHER SENSOR DATA COMPLETENESS
   sleep_level         :  27.15% complete (14,418,576 of 53,104,440 records)
   screen              :  99.46% complete (52,819,380 of 53,104,440 records)

3. DEPRESSION LABEL COMPLETENESS
   depression_status_baseline         :  97.85% (51,962,700 records)
   depression_status_endpoint         :  70.73% (37,561,560 records)
   depression_trajectory              :  

## Step 12: Analyze Study Duration and Temporal Coverage

In [12]:
print("\n" + "="*100)
print("STUDY DURATION AND TEMPORAL COVERAGE ANALYSIS")
print("="*100)

print(f"\n1. OVERALL STUDY PERIOD")
print(f"   {'='*80}")
min_date = df['dateTime'].min()
max_date = df['dateTime'].max()
total_duration = (max_date - min_date).days
print(f"   Study start date:           {min_date}")
print(f"   Study end date:             {max_date}")
print(f"   Total duration:             {total_duration} days")

print(f"\n2. PARTICIPANT-LEVEL DURATION STATISTICS (in days)")
print(f"   {'='*80}")
duration_stats = study_info['duration_days'].describe()
print(f"   Mean:                       {duration_stats['mean']:6.1f} days")
print(f"   Median:                     {duration_stats['50%']:6.1f} days")
print(f"   Std Dev:                    {duration_stats['std']:6.1f} days")
print(f"   Min:                        {duration_stats['min']:6.0f} days")
print(f"   Max:                        {duration_stats['max']:6.0f} days")
print(f"   25th percentile:            {duration_stats['25%']:6.1f} days")
print(f"   75th percentile:            {duration_stats['75%']:6.1f} days")

print(f"\n3. MINUTE-LEVEL RECORDS PER PARTICIPANT")
print(f"   {'='*80}")
records_stats = study_info['actual_count'].describe()
print(f"   Mean:                       {records_stats['mean']:,.0f} records")
print(f"   Median:                     {records_stats['50%']:,.0f} records")
print(f"   Std Dev:                    {records_stats['std']:,.0f} records")
print(f"   Min:                        {records_stats['min']:,.0f} records")
print(f"   Max:                        {records_stats['max']:,.0f} records")

print(f"\n4. TEMPORAL DISTRIBUTION")
print(f"   {'='*80}")
month_dist = df['dateTime'].dt.to_period('M').value_counts().sort_index()
print(f"\n   Records by month:")
for month, count in month_dist.items():
    print(f"   {month}: {count:,} records")

print(f"\n5. PARTICIPANT RECRUITMENT AND DROPOUT")
print(f"   {'='*80}")
participant_start = df.groupby('pid')['dateTime'].min().value_counts().sort_index()
first_week_starts = len(participant_start[participant_start.index == participant_start.index.min()])
print(f"   First participant start date: {participant_start.index.min()}")
print(f"   Last participant start date:  {participant_start.index.max()}")
print(f"   Participants starting week 1: {first_week_starts} out of {df['pid'].nunique()}")

print(f"\n✓ Temporal analysis complete")


STUDY DURATION AND TEMPORAL COVERAGE ANALYSIS

1. OVERALL STUDY PERIOD
   Study start date:           2021-09-07 00:00:00
   Study end date:             2023-01-17 23:59:00
   Total duration:             497 days

2. PARTICIPANT-LEVEL DURATION STATISTICS (in days)
   Mean:                        223.3 days
   Median:                      224.0 days
   Std Dev:                      94.7 days
   Min:                            11 days
   Max:                           491 days
   25th percentile:             200.2 days
   75th percentile:             237.8 days

3. MINUTE-LEVEL RECORDS PER PARTICIPANT
   Mean:                       319,906 records
   Median:                     322,500 records
   Std Dev:                    136,330 records
   Min:                        15,840 records
   Max:                        706,980 records

4. TEMPORAL DISTRIBUTION

   Records by month:
   2021-09: 1,908,000 records
   2021-10: 4,769,280 records
   2021-11: 4,711,680 records
   2021-12: 5,768,64

## Step 13: Generate Final Summary Report

In [13]:
print("\n" + "="*100)
print("FINAL SUMMARY REPORT")
print("="*100)

# Verify output file exists
if not OUTPUT_FILE.exists():
    print(f"\n✗ ERROR: Output file not found at {OUTPUT_FILE}")
else:
    file_size_gb = OUTPUT_FILE.stat().st_size / (1024**3)
    
    # Get depression label counts
    pid_labels = df[['pid', 'depression_status_baseline', 'depression_status_endpoint', 'depression_trajectory']].drop_duplicates('pid')
    
    # Calculate study statistics
    study_info = df.groupby('pid')['dateTime'].agg(['min', 'max', 'count']).reset_index()
    study_info.columns = ['pid', 'start_date', 'end_date', 'record_count']
    study_info['duration_days'] = (study_info['end_date'] - study_info['start_date']).dt.days + 1
    study_info['expected_records'] = study_info['duration_days'] * 24 * 60
    study_info['recovery_rate'] = (study_info['record_count'] / study_info['expected_records'] * 100).fillna(0)
    
    # Prepare summary information
    sensor_cols = ['HR', 'Steps', 'Floors', 'Fairly_Active', 'Lightly_Active', 'Sedentary', 'Very_Active']
    sensor_cols = [c for c in sensor_cols if c in df.columns]
    
    summary_report = f"""
{'='*100}
✓ HRD_RAW_MinuteLevel.csv SUCCESSFULLY CREATED
{'='*100}

OUTPUT LOCATION: {OUTPUT_FILE}
FILE SIZE: {file_size_gb:.3f} GB
CREATION DATE: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}

{'='*80}
DATASET STRUCTURE
{'='*80}

RESOLUTION:      Minute-level (1 record = 1 calendar minute)
TOTAL RECORDS:   {len(df):,} minute-level observations
TOTAL COLUMNS:   {len(df.columns)}
PARTICIPANTS:    {df['pid'].nunique()}
DATE RANGE:      {df['dateTime'].min().date()} to {df['dateTime'].max().date()}
DURATION:        {(df['dateTime'].max() - df['dateTime'].min()).days} days

{'='*80}
DATA COMPONENTS
{'='*80}

1. FITBIT SENSOR DATA (Minute-level):
   • Heart Rate (HR) - beats per minute
   • Steps - step count per minute
   • Floors - floors climbed per minute
   • Activity Levels:
     - Sedentary: minutes with sedentary activity
     - Lightly_Active: minutes with light activity
     - Fairly_Active: minutes with fairly intense activity
     - Very_Active: minutes with very intense activity

2. SLEEP DATA (Minute-level):
   • sleep_level: Sleep stage classification
     Values: wake, light, deep, REM, unknown
   • Expanded from intraday sleep records (start + duration → minute records)

3. MOBILE SENSING (AWARE):
   • screen: Screen state (0=locked, 1=unlocked)
     Uses forward-fill method: each minute inherits last known state
   • Calls: Available in source, optional for future analysis

4. DEPRESSION ASSESSMENT (CES-D-Based):
   • depression_status_baseline: Binary status at baseline (0=healthy, 1=depressed)
   • depression_status_endpoint: Binary status at endpoint (0=healthy, 1=depressed)
   • ces_d_baseline_score: Continuous CES-D score at baseline (0-60 range)
   • ces_d_endpoint_score: Continuous CES-D score at endpoint (0-60 range)
   • depression_trajectory: Longitudinal classification
     - Stable_Healthy: healthy at both timepoints (0→0)
     - Onset: developed depression (0→1)
     - Recovery: remission from depression (1→0)
     - Persistent: depressed throughout (1→1)
     - Unknown: incomplete CES-D data
   
   Threshold: CES-D ≥16 indicates depression
   Applied to all minute-level records per participant

{'='*80}
DEPRESSION LABEL STATISTICS
{'='*80}

BASELINE (Pre-Study Assessment):
  Healthy (CES-D < 16):   {(pid_labels['depression_status_baseline'] == 0).sum():4d} participants
  Depressed (CES-D ≥ 16): {(pid_labels['depression_status_baseline'] == 1).sum():4d} participants
  Unknown:                {pid_labels['depression_status_baseline'].isna().sum():4d} participants
  Mean CES-D Score:       {df['ces_d_baseline_score'].mean():6.2f} (SD: {df['ces_d_baseline_score'].std():.2f})

ENDPOINT (Post-Study Assessment):
  Healthy (CES-D < 16):   {(pid_labels['depression_status_endpoint'] == 0).sum():4d} participants
  Depressed (CES-D ≥ 16): {(pid_labels['depression_status_endpoint'] == 1).sum():4d} participants
  Unknown:                {pid_labels['depression_status_endpoint'].isna().sum():4d} participants
  Mean CES-D Score:       {df['ces_d_endpoint_score'].mean():6.2f} (SD: {df['ces_d_endpoint_score'].std():.2f})

LONGITUDINAL TRAJECTORIES:
  Stable_Healthy (0→0):   {(pid_labels['depression_trajectory'] == 'Stable_Healthy').sum():4d} (remained healthy)
  Onset (0→1):            {(pid_labels['depression_trajectory'] == 'Onset').sum():4d} (developed depression)
  Recovery (1→0):         {(pid_labels['depression_trajectory'] == 'Recovery').sum():4d} (recovered from depression)
  Persistent (1→1):       {(pid_labels['depression_trajectory'] == 'Persistent').sum():4d} (remained depressed)
  Unknown:                {pid_labels['depression_trajectory'].isna().sum():4d} (incomplete data)

{'='*80}
DATA COMPLETENESS SUMMARY
{'='*80}

FITBIT SENSORS:
"""

    # Add sensor completeness
    for col in sensor_cols:
        completeness = (df[col].notna().sum() / len(df)) * 100
        summary_report += f"  {col:25s}: {completeness:6.2f}%\n"
    
    summary_report += f"""
OTHER STREAMS:
  sleep_level             : {(df['sleep_level'].notna().sum() / len(df) * 100):6.2f}%
  screen                  : {(df['screen'].notna().sum() / len(df) * 100):6.2f}%

DEPRESSION LABELS:
  depression_status_baseline  : {(df['depression_status_baseline'].notna().sum() / len(df) * 100):6.2f}%
  depression_status_endpoint  : {(df['depression_status_endpoint'].notna().sum() / len(df) * 100):6.2f}%
  depression_trajectory       : {(df['depression_trajectory'].notna().sum() / len(df) * 100):6.2f}%

OVERALL DATA RECOVERY RATE: {(len(df) / study_info['expected_records'].sum() * 100):.1f}%

{'='*80}
PARTICIPANT STATISTICS
{'='*80}

Duration per Participant (days):
  Mean:                   {study_info['duration_days'].mean():8.1f}
  Median:                 {study_info['duration_days'].median():8.1f}
  Std Dev:                {study_info['duration_days'].std():8.1f}
  Range:                  {study_info['duration_days'].min():.0f} - {study_info['duration_days'].max():.0f}

Records per Participant:
  Mean:                   {study_info['record_count'].mean():8,.0f}
  Median:                 {study_info['record_count'].median():8,.0f}
  Range:                  {study_info['record_count'].min():,.0f} - {study_info['record_count'].max():,.0f}

Data Recovery Rate:
  Mean:                   {study_info['recovery_rate'].mean():8.1f}%
  Median:                 {study_info['recovery_rate'].median():8.1f}%
  Min:                    {study_info['recovery_rate'].min():8.1f}%
  Max:                    {study_info['recovery_rate'].max():8.1f}%

{'='*80}
COLUMN INFORMATION ({len(df.columns)} columns total)
{'='*80}

"""
    
    # Add detailed column info
    for idx, col in enumerate(df.columns, 1):
        dtype_str = str(df[col].dtype)
        non_null = df[col].notna().sum()
        completeness = (non_null / len(df)) * 100
        summary_report += f"{idx:2d}. {col:30s} | dtype: {dtype_str:12s} | {non_null:10,} ({completeness:5.1f}%)\n"
    
    summary_report += f"""
{'='*80}
NEXT STEPS / RESEARCH APPLICATIONS
{'='*80}

✓ Longitudinal analysis: Compare rhythm changes from pre to post-study
✓ Group comparisons: Healthy vs depressed populations
✓ Subgroup analysis: Onset, Recovery, Persistent depression groups
✓ Circadian rhythm analysis: Daily patterns of HR, activity, screen use, sleep
✓ Predictive modeling: Early indicators of depression status change
✓ Correlational studies: Sleep-activity-mood relationships
✓ Time-series analysis: Temporal patterns and changepoints
✓ Participant segmentation: Clustering based on behavioral patterns

{'='*80}
DATASET READY FOR ANALYSIS
{'='*80}

All sensor data and depression labels are:
  ✓ Integrated at minute-level resolution
  ✓ Aligned by participant and time
  ✓ Quality-checked and formatted
  ✓ Ready for comprehensive rhythm analysis and mental health research

Use HRD_RAW_MinuteLevel.csv for all downstream analyses.
"""
    
    print(summary_report)


FINAL SUMMARY REPORT

✓ HRD_RAW_MinuteLevel.csv SUCCESSFULLY CREATED

OUTPUT LOCATION: c:\Users\umroot\Desktop\Human-Rhythms-Dataset\HRD\HRD_RAW_MinuteLevel.csv
FILE SIZE: 4.025 GB
CREATION DATE: 2026-05-09 19:19:57

DATASET STRUCTURE

RESOLUTION:      Minute-level (1 record = 1 calendar minute)
TOTAL RECORDS:   53,104,440 minute-level observations
TOTAL COLUMNS:   17
PARTICIPANTS:    166
DATE RANGE:      2021-09-07 to 2023-01-17
DURATION:        497 days

DATA COMPONENTS

1. FITBIT SENSOR DATA (Minute-level):
   • Heart Rate (HR) - beats per minute
   • Steps - step count per minute
   • Floors - floors climbed per minute
   • Activity Levels:
     - Sedentary: minutes with sedentary activity
     - Lightly_Active: minutes with light activity
     - Fairly_Active: minutes with fairly intense activity
     - Very_Active: minutes with very intense activity

2. SLEEP DATA (Minute-level):
   • sleep_level: Sleep stage classification
     Values: wake, light, deep, REM, unknown
   • Expan

# Step 14: Build Daily-Level Survey Dataset

In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 14: BUILD DAILY-LEVEL SURVEY DATASET WITH DEPRESSION LABELS
# ═══════════════════════════════════════════════════════════════════════════════
# This cell is completely independent and reads all required data itself

import pandas as pd
import numpy as np
from pathlib import Path
import re
import time

# ─────────────────────────────────────────────────────────────────────────────
# INITIALIZATION: Define paths (if not already defined)
# ─────────────────────────────────────────────────────────────────────────────
try:
    # If variables are defined from previous cells, use them
    _ = HRD_DIR
except NameError:
    # If not defined, define them
    HRD_DIR = Path(r"c:\Users\umroot\Desktop\Human-Rhythms-Dataset\HRD")
    CLEANED_RAW = HRD_DIR / "Cleaned_Raw_Data"
    OUTPUT_FILE = HRD_DIR / "HRD_RAW_MinuteLevel.csv"
    SURVEY_OUTPUT = HRD_DIR / "HRD_Survey_Dailylevel.csv"

print("="*100)
print("STEP 14: BUILD DAILY-LEVEL SURVEY DATASET WITH DEPRESSION LABELS")
print("="*100)
print(f"\nData source: {HRD_DIR}")
print(f"Output file: {SURVEY_OUTPUT}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Load Depression Labels from HRD_RAW_MinuteLevel.csv
# ─────────────────────────────────────────────────────────────────────────────
print("\n[1/5] Loading depression labels from HRD_RAW_MinuteLevel.csv...")

if not OUTPUT_FILE.exists():
    raise FileNotFoundError(f"HRD_RAW_MinuteLevel.csv not found at {OUTPUT_FILE}")

try:
    # Retry logic for file locks (file may be temporarily locked)
    max_retries = 5
    retry_delay = 2  # seconds
    minute_df = None
    
    for attempt in range(max_retries):
        try:
            minute_df = pd.read_csv(OUTPUT_FILE, 
                                   usecols=['pid', 'dateTime', 'depression_status_baseline', 
                                           'depression_status_endpoint', 'depression_trajectory',
                                           'ces_d_baseline_score', 'ces_d_endpoint_score'],
                                   dtype={'pid': str})
            break  # Success - exit retry loop
        except PermissionError as pe:
            if attempt < max_retries - 1:
                print(f"  ⚠ File lock detected, retrying in {retry_delay} seconds (attempt {attempt + 1}/{max_retries})...")
                time.sleep(retry_delay)
            else:
                raise pe  # Re-raise on final attempt
    
    if minute_df is None:
        raise RuntimeError("Failed to load CSV after retries")
    
    minute_df['dateTime'] = pd.to_datetime(minute_df['dateTime'])
    minute_df['date'] = minute_df['dateTime'].dt.date
    
    # Extract unique (pid, date) pairs
    unique_days = minute_df.groupby('pid')['date'].unique()
    
    print(f"  ✓ Loaded {len(minute_df):,} minute-level records")
    print(f"  ✓ Found {len(unique_days)} participants")
    print(f"  ✓ Total unique dates: {minute_df['date'].nunique()}")
    
    # Create depression_labels dictionary
    depression_labels = {}
    for pid in minute_df['pid'].unique():
        pid_data = minute_df[minute_df['pid'] == pid].iloc[0]
        depression_labels[pid] = {
            'depression_status_baseline': pid_data['depression_status_baseline'],
            'depression_status_endpoint': pid_data['depression_status_endpoint'],
            'depression_trajectory': pid_data['depression_trajectory'],
            'ces_d_baseline_score': pid_data['ces_d_baseline_score'],
            'ces_d_endpoint_score': pid_data['ces_d_endpoint_score']
        }
    
    print(f"  ✓ Extracted depression labels for {len(depression_labels)} participants")
    
except Exception as e:
    raise RuntimeError(f"Error loading depression labels: {e}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Find and Load Survey Files
# ─────────────────────────────────────────────────────────────────────────────
print("\n[2/5] Loading and mapping daily survey responses...")

DAILY_SURVEY_DIR = CLEANED_RAW / "Surveys" / "Daily_Surveys"

if not DAILY_SURVEY_DIR.exists():
    raise FileNotFoundError(f"Daily_Surveys directory not found at {DAILY_SURVEY_DIR}")

survey_files = sorted(DAILY_SURVEY_DIR.glob('*.csv'))
print(f"  ✓ Found {len(survey_files)} survey files")

survey_mapping = {}  # (pid, date) → {survey_columns: values}
survey_columns = set()
failed_pids = []

for survey_file in survey_files:
    pid = survey_file.stem.lower()
    
    try:
        survey_df = pd.read_csv(survey_file, dtype={'PID': str})
        
        # Extract EndDate and convert to date object
        if 'EndDate' in survey_df.columns:
            survey_df['EndDate'] = pd.to_datetime(survey_df['EndDate'])
            survey_df['date'] = survey_df['EndDate'].dt.date
        else:
            print(f"  ⚠ No EndDate column in {pid}. Skipping survey file.")
            failed_pids.append(pid)
            continue
        
        # Store survey responses mapped by (pid, date)
        for _, row in survey_df.iterrows():
            survey_key = (pid, row['date'])
            response_data = {}
            
            # Extract all columns except metadata
            exclude_cols = {'PID', 'ResponseID', 'StartDate', 'EndDate'}
            for col in survey_df.columns:
                if col not in exclude_cols:
                    response_data[col] = row[col]
                    survey_columns.add(col)
            
            survey_mapping[survey_key] = response_data
    
    except Exception as e:
        print(f"  ⚠ Error reading survey file {pid}: {e}")
        failed_pids.append(pid)

survey_columns = sorted(list(survey_columns))
print(f"  ✓ Mapped {len(survey_mapping)} (participant, date) pairs")
print(f"  ✓ Found {len(survey_columns)} unique survey questions")
if failed_pids:
    print(f"  ⚠ Failed to load surveys for: {', '.join(failed_pids[:5])}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Create Daily DataFrame by merging depression labels with surveys
# ─────────────────────────────────────────────────────────────────────────────
print("\n[3/5] Creating daily-level dataset...")

depression_cols = ['depression_status_baseline', 'depression_status_endpoint', 
                   'depression_trajectory', 'ces_d_baseline_score', 'ces_d_endpoint_score']

daily_records = []

# Iterate through unique (pid, date) pairs from depression data
unique_days = minute_df.groupby('pid')['date'].unique()

for pid in unique_days.index:
    dates = unique_days[pid]
    
    for date in dates:
        record = {
            'PID': pid,
            'DATE': date
        }
        
        # Add depression labels
        if pid in depression_labels:
            for dep_col in depression_cols:
                record[dep_col] = depression_labels[pid][dep_col]
        else:
            for dep_col in depression_cols:
                record[dep_col] = np.nan
        
        # Add survey responses
        survey_key = (pid, date)
        if survey_key in survey_mapping:
            for survey_col in survey_columns:
                record[survey_col] = survey_mapping[survey_key].get(survey_col, np.nan)
        else:
            for survey_col in survey_columns:
                record[survey_col] = np.nan
        
        daily_records.append(record)

daily_survey_df = pd.DataFrame(daily_records)

print(f"  ✓ Created DataFrame with {len(daily_survey_df):,} rows")
print(f"  ✓ {len(daily_survey_df.columns)} total columns")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3b: Rename Survey Column Names to Shorter, Readable Format
# ─────────────────────────────────────────────────────────────────────────────
print("\n[3b/5] Renaming survey columns to shorter format...")

# Define explicit mappings for known survey questions
rename_map = {
    'Q2: Sleep Quality': 'SleepQuality',
    'Q3: Cognitive Energy (1-Unfocused, 5-Focused)': 'CognitiveEnergy',
    'Q4: Physical Energy': 'PhysicalEnergy',
    'Q5: Emotional Energy': 'EmotionalEnergy',
    'Q6_5: How much did you engage in the following activities?-Professional activities (e.g., work, school)': 'ProfessionalActivities',
    'Q6_6: How much did you engage in the following activities?-Social activities (e.g., spending time with family and friends, dating, party, going to movies/concert)': 'SocialActivities',
    'Q6_7: How much did you engage in the following activities?-Physical activities (e.g., exercise, walking, hiking, sports)': 'PhysicalActivities',
    'Q6_8: How much did you engage in the following activities?-Other leisure (e.g., reading books, playing games, watching TV/movies, shopping, housework, cooking, relaxation)': 'OtherLeisure',
    'Q7: How mentally demanding were your tasks? (1-low to 5-high)': 'MentalDemand',
    'Q8: How physically demanding were your tasks? (1-low to 5-high)': 'PhysicalDemand',
    'Q9: How hurried or rushed was the pace of the tasks? (1-low to 5-high)': 'RushedPace'
}

def shorten_survey_column_name(col_name):
    """Convert long survey column names to shorter, readable format."""
    # Check explicit rename map first
    if col_name in rename_map:
        return rename_map[col_name]
    
    # Handle Q6 activity questions (format: "Q6_X: How much... - Activity name")
    if 'Q6' in col_name and '-' in col_name:
        activity_part = col_name.split('-', 1)[1].strip()
        # Remove parenthetical examples and get main activity name
        activity_text = activity_part.split('(')[0].strip()
        words = activity_text.replace('?', '').split()
        short_name = ''.join(word.capitalize() for word in words[:3])
        return short_name if short_name else col_name
    
    # Fallback: Try to extract meaningful parts
    if 'Q' in col_name and ':' in col_name:
        q_part = col_name.split(':')[0].strip()
        text_part = col_name.split(':', 1)[1].strip()
        # Remove parenthetical descriptions
        text_part = text_part.split('(')[0].strip()
        words = text_part.replace('?', '').split()
        short_name = ''.join(word.capitalize() for word in words[:3])
        return f"{q_part}_{short_name}" if short_name else q_part
    
    return col_name

# Build rename mapping and apply
survey_rename_map = {}
for col in daily_survey_df.columns:
    if col not in ['PID', 'DATE'] + depression_cols:
        col_str = str(col)
        survey_rename_map[col_str] = shorten_survey_column_name(col_str)

# Rename columns
daily_survey_df = daily_survey_df.rename(columns=survey_rename_map)

# Reorder columns: PID, DATE, depression columns, then survey columns
non_survey_cols = set(['PID', 'DATE'] + depression_cols)
survey_cols_renamed = [col for col in daily_survey_df.columns if col not in non_survey_cols]
survey_cols_renamed = sorted(survey_cols_renamed)

final_columns = ['PID', 'DATE'] + depression_cols + survey_cols_renamed
daily_survey_df = daily_survey_df[final_columns]

print(f"  ✓ Renamed {len(survey_rename_map)} survey columns")
print(f"  ✓ Final structure: {len(daily_survey_df.columns)} columns")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: Save Daily-Level Dataset to CSV
# ─────────────────────────────────────────────────────────────────────────────
print("\n[4/5] Saving daily-level dataset to CSV...")

try:
    daily_survey_df.to_csv(SURVEY_OUTPUT, index=False)
    file_size_gb = SURVEY_OUTPUT.stat().st_size / (1024**3)
    print(f"  ✓ Saved to {SURVEY_OUTPUT}")
    print(f"  ✓ File size: {file_size_gb:.3f} GB")
except Exception as e:
    print(f"  ✗ Error saving file: {e}")
    raise

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: Summary Statistics
# ─────────────────────────────────────────────────────────────────────────────
print("\n[5/5] Summary Statistics")
print("─" * 100)

# Depression label coverage
dep_baseline_count = int(daily_survey_df['depression_status_baseline'].notna().sum().item())
dep_endpoint_count = int(daily_survey_df['depression_status_endpoint'].notna().sum().item())
dep_trajectory_count = int(daily_survey_df['depression_trajectory'].notna().sum().item())
depressed_count = int((daily_survey_df['depression_status_endpoint'] == 1).sum().item())

print("\nDepression Label Coverage:")
print(f"  • Baseline status:    {dep_baseline_count:6,} ({100*dep_baseline_count/len(daily_survey_df):5.1f}%)")
print(f"  • Endpoint status:    {dep_endpoint_count:6,} ({100*dep_endpoint_count/len(daily_survey_df):5.1f}%)")
print(f"  • Trajectory status:  {dep_trajectory_count:6,} ({100*dep_trajectory_count/len(daily_survey_df):5.1f}%)")
print(f"  • Depressed (endpoint=1): {depressed_count:,} ({100*depressed_count/len(daily_survey_df):5.1f}%)")

# Survey response coverage (sample of first 5 columns)
print("\nSurvey Question Coverage (first 5 columns):")
survey_q_cols = [col for col in final_columns if col not in ['PID', 'DATE'] + depression_cols][:5]
for col in survey_q_cols:
    col_str = str(col)
    col_coverage = int(daily_survey_df[col_str].notna().sum().item())
    col_pct = 100 * col_coverage / len(daily_survey_df)
    print(f"  • {col_str:30s}: {col_coverage:6,} ({col_pct:5.1f}%)")

# Final structure info
print(f"\nFinal Dataset Structure:")
print(f"  • Total rows:     {len(daily_survey_df):,}")
print(f"  • Total columns:  {len(daily_survey_df.columns)}")
print(f"  • Column types:   {daily_survey_df.dtypes.value_counts().to_dict()}")

print("\n" + "="*100)
print("✓ STEP 14 COMPLETE: Daily-level survey dataset created successfully!")
print("="*100)


STEP 14: BUILD DAILY-LEVEL SURVEY DATASET WITH DEPRESSION LABELS

Data source: c:\Users\umroot\Desktop\Human-Rhythms-Dataset\HRD
Output file: c:\Users\umroot\Desktop\Human-Rhythms-Dataset\HRD\HRD_Survey_Dailylevel.csv

[1/5] Loading depression labels from HRD_RAW_MinuteLevel.csv...
  ✓ Loaded 53,104,440 minute-level records
  ✓ Found 166 participants
  ✓ Total unique dates: 498
  ✓ Extracted depression labels for 166 participants

[2/5] Loading and mapping daily survey responses...
  ✓ Found 165 survey files
  ✓ Mapped 25086 (participant, date) pairs
  ✓ Found 12 unique survey questions

[3/5] Creating daily-level dataset...
  ✓ Created DataFrame with 36,884 rows
  ✓ 19 total columns

[3b/5] Renaming survey columns to shorter format...
  ✓ Renamed 12 survey columns
  ✓ Final structure: 19 columns

[4/5] Saving daily-level dataset to CSV...
  ✓ Saved to c:\Users\umroot\Desktop\Human-Rhythms-Dataset\HRD\HRD_Survey_Dailylevel.csv
  ✓ File size: 0.003 GB

[5/5] Summary Statistics
─────────